# Public REST market data

Required environment variables: none.

This notebook uses the production Environment preset and synchronous REST methods.

It also demonstrates handling API, rate-limit, and transport failures for a public REST read.

In [ ]:
from exchangeos_sdk import (
    APIError,
    ClientOptions,
    ExchangeOSClient,
    RateLimitError,
    RestMarketType,
    TransportError,
)

def load_public_market_data():
    try:
        with ExchangeOSClient(options=ClientOptions(preserve_raw_response=True)) as client:
            spot = client.market.instruments(market_type=RestMarketType.SPOT)
            perps = client.market.instruments(market_type=RestMarketType.PERP)
            tickers = client.market.tickers(market_type=RestMarketType.PERP)
        return spot, perps, tickers
    except RateLimitError as error:
        print(f'Retry after {error.retry_after or 1.0}s (request_id={error.request_id})')
    except APIError as error:
        print(f'ExchangeOS API error: {error.code} {error.message} (request_id={error.request_id})')
    except TransportError as error:
        print(f'Transport error for {error.method} {error.path}: {error}')
    return None

market_data = load_public_market_data()
if market_data is not None:
    spot, perps, tickers = market_data
    print({"spotInstruments": len(spot), "perpInstruments": len(perps), "tickers": len(tickers)})
    if tickers:
        ticker = tickers[0]
        print({"assetId": ticker.asset_id, "lastPx": ticker.last_px})